In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "4"  # silences the KMeans memory-leak warning on Windows

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, silhouette_score
from sklearn.model_selection import (
    GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold,
    cross_val_predict, cross_val_score, train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import calibration_and_holdout_data, summary_data_from_transaction_data

ROOT = Path(r"C:\Users\reinh\Downloads\Projects\customer-segmentation-churn")
RAW_UCI = r"C:\Users\reinh\Downloads\Projects\Online Retail II UCI/online_retail_II.csv"
RAW_TELCO = r"C:\Users\reinh\Downloads\Projects\Telecom Customer Churn\WA_Fn-UseC_-Telco-Customer-Churn.csv"
CLEAN =  r"C:\Users\reinh\Downloads\Projects\Output CSVs\online_retail_clean.csv"
RFM_OUT = r"C:\Users\reinh\Downloads\Projects\Output CSVs\rfm_table.csv"
SCORES = r"C:\Users\reinh\Downloads\Projects\Output CSVs\customer_scores.csv"
CLV_OUT = r"C:\Users\reinh\Downloads\Projects\Output CSVs\clv_table_cutoff.csv"
FINAL = r"C:\Users\reinh\Downloads\Projects\Output CSVs\customer_scores_final.csv"

ORDER_CAP = 50_000  # invoices above this are capped for CLV only

In [2]:
# %% 1. Load, pair cancellations (including partial), exclude non-product lines, and clean
from collections import defaultdict

raw = pd.read_csv(RAW_UCI, encoding="ISO-8859-1", parse_dates=["InvoiceDate"])
print(f"Loaded:                         {len(raw):>10,}")

raw = raw[raw["Customer ID"].notna()].copy()
raw["Customer ID"] = raw["Customer ID"].astype(int)
raw["Invoice"] = raw["Invoice"].astype(str)
raw["StockCode"] = raw["StockCode"].astype(str).str.strip().str.upper()
raw["Price"] = raw["Price"].round(2)
print(f"With Customer ID:               {len(raw):>10,}")

is_cancel = raw["Invoice"].str.startswith("C")
cancels = raw[is_cancel & (raw["Quantity"] < 0)]
sales = raw[~is_cancel & (raw["Quantity"] > 0) & (raw["Price"] > 0)].copy()
print(f"Cancellation lines:             {len(cancels):>10,}")

# Remaining quantity on each sale line, and an index of sales by customer and product
remaining = dict(zip(sales.index, sales["Quantity"]))
by_key = defaultdict(list)     # (customer, stock, price) -> [(date, row)], latest first
by_cs = defaultdict(list)      # (customer, stock) -> [(date, price, row)]
for idx, r in sales.iterrows():
    by_key[(r["Customer ID"], r["StockCode"], r["Price"])].append((r["InvoiceDate"], idx))
    by_cs[(r["Customer ID"], r["StockCode"])].append((r["InvoiceDate"], r["Price"], idx))
for k in by_key:
    by_key[k].sort(reverse=True)

# Match each cancellation to earlier sales of the same customer, product and price,
# latest first, reducing quantities as they are used
full, partial, unmatched_reasons = 0, 0, defaultdict(list)
for idx, r in cancels.sort_values("InvoiceDate").iterrows():
    need = -r["Quantity"]
    wanted = need
    key = (r["Customer ID"], r["StockCode"], r["Price"])
    for d, oidx in by_key.get(key, []):
        if d > r["InvoiceDate"] or remaining[oidx] <= 0:
            continue
        take = min(need, remaining[oidx])
        remaining[oidx] -= take
        need -= take
        if need == 0:
            break
    if need == 0:
        full += 1
    elif need < wanted:
        partial += 1
    else:
        # Nothing matched: work out why
        cs = [(d, p, o) for d, p, o in by_cs.get((r["Customer ID"], r["StockCode"]), [])
              if d <= r["InvoiceDate"]]
        if not cs:
            reason = "no earlier sale of this product to this customer"
        elif any(p != r["Price"] for _, p, _ in cs):
            reason = "earlier sale at a different price"
        else:
            reason = "earlier sales already used up"
        unmatched_reasons[reason].append(r["InvoiceDate"])

unmatched_total = sum(len(v) for v in unmatched_reasons.values())
print(f"  Fully paired:                 {full:>10,}")
print(f"  Partly paired:                {partial:>10,}")
print(f"  Unmatched:                    {unmatched_total:>10,}")
for reason, dates in unmatched_reasons.items():
    s = pd.Series(dates)
    print(f"    - {reason}: {len(dates):,}  (dates {s.min().date()} to {s.max().date()})")

# Apply the reduced quantities to sales lines
sales["Quantity"] = sales.index.map(remaining)
sales = sales[sales["Quantity"] > 0]
print(f"\nSales lines after pairing:      {len(sales):>10,}")

# Remove non-product lines (postage, manual adjustments, bank charges, fees)
NON_PRODUCT = {"POST", "D", "M", "C2", "DOT", "BANK CHARGES", "AMAZONFEE",
               "ADJUST", "ADJUST2", "CRUK", "PADS", "S", "B"}
np_lines = sales[sales["StockCode"].isin(NON_PRODUCT)]
print(f"Non-product lines removed:      {len(np_lines):>10,}  "
      f"(£{(np_lines['Quantity'] * np_lines['Price']).sum():,.0f})")
sales = sales[~sales["StockCode"].isin(NON_PRODUCT)]

# Invoice 581483: confirm it has been paired; exclude if still present
EXCLUDE_INVOICES = {"581483"}
still_there = sales["Invoice"].isin(EXCLUDE_INVOICES).sum()
sales = sales[~sales["Invoice"].isin(EXCLUDE_INVOICES)].copy()
print(f"Invoice 581483 lines still present before exclusion: {still_there}")

clean = sales.copy()
clean["Revenue"] = clean["Quantity"] * clean["Price"]
df = clean  # the rest of the notebook reads df

print(f"\nClean rows:                     {len(clean):>10,}")
print(f"Clean customers:                {clean['Customer ID'].nunique():>10,}")
clean.to_csv(CLEAN, index=False)
print(f"Saved to {CLEAN}")

Loaded:                          1,067,371
With Customer ID:                  824,364
Cancellation lines:                 18,744
  Fully paired:                     15,562
  Partly paired:                       119
  Unmatched:                         3,063
    - no earlier sale of this product to this customer: 2,070  (dates 2009-12-01 to 2011-12-09)
    - earlier sales already used up: 127  (dates 2009-12-03 to 2011-12-06)
    - earlier sale at a different price: 866  (dates 2009-12-10 to 2011-12-08)

Sales lines after pairing:         799,385
Non-product lines removed:           2,795  (£232,370)
Invoice 581483 lines still present before exclusion: 0

Clean rows:                        796,590
Clean customers:                     5,839
Saved to C:\Users\reinh\Downloads\Projects\Output CSVs\online_retail_clean.csv


In [9]:
# %% Check the removed non-product lines
np_lines = np_lines.assign(Value=np_lines["Quantity"] * np_lines["Price"])

by_code = (
    np_lines.groupby("StockCode")
    .agg(Lines=("Invoice", "size"),
         Value=("Value", "sum"),
         Descriptions=("Description", lambda s: " | ".join(sorted(s.dropna().unique())[:3])))
    .sort_values("Value", ascending=False)
)
print(by_code.round(2).to_string())

print("\nLargest individual removed lines:")
print(np_lines.nlargest(15, "Value")[
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Value"]
].to_string())

              Lines      Value                                                               Descriptions
StockCode                                                                                                
POST           1819  123681.96                                                                    POSTAGE
M               643   79133.22                                                                     Manual
C2              248   12546.00                                                                   CARRIAGE
DOT              16   11906.36                                                             DOTCOM POSTAGE
ADJUST           32    3538.52  Adjustment by john on 26/01/2010 16 | Adjustment by john on 26/01/2010 17
ADJUST2           3     731.05                                        Adjustment by Peter on Jun 25 2010 
BANK CHARGES     29     435.00                                                Bank Charges | Bank Charges
D                 5     397.89                

In [4]:
# %% 2. RFM segmentation on the full period (exploratory)
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = pd.DataFrame({
    "Recency": (snapshot - df.groupby("Customer ID")["InvoiceDate"].max()).dt.days,
    "Frequency": df.groupby("Customer ID")["Invoice"].nunique(),
    "Monetary": df.groupby("Customer ID")["Revenue"].sum(),
})
print(f"Customers: {len(rfm):,}")

X_full = StandardScaler().fit_transform(np.log1p(rfm[["Recency", "Frequency", "Monetary"]]))

print("\nSilhouette scores:")
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_full)
    print(f"  k={k}: {silhouette_score(X_full, labels, sample_size=5000, random_state=42):.3f}")

rfm["Cluster"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_full)
print("\nCluster profile (k=4):")
print(rfm.groupby("Cluster").agg(
    Customers=("Recency", "size"),
    Recency=("Recency", "mean"),
    Frequency=("Frequency", "mean"),
    Monetary=("Monetary", "mean"),
).round(1))

rfm.to_csv(RFM_OUT)
print(f"Saved RFM table ({len(rfm):,} customers) to {RFM_OUT}")

Customers: 5,839

Silhouette scores:
  k=2: 0.436
  k=3: 0.346
  k=4: 0.364
  k=5: 0.343
  k=6: 0.333
  k=7: 0.298
  k=8: 0.297

Cluster profile (k=4):
         Customers  Recency  Frequency  Monetary
Cluster                                         
0             1250     28.2        3.0     849.7
1             1451    228.1        5.0    1899.5
2             1178     27.8       19.1   10465.1
3             1960    393.4        1.4     317.1
Saved RFM table (5,839 customers) to C:\Users\reinh\Downloads\Projects\Output CSVs\rfm_table.csv


In [5]:
# %% 3. Churn model with a time cutoff, segments, and validation
END = df["InvoiceDate"].max()
CUTOFF = END - pd.Timedelta(days=90)
print(f"Features use data up to {CUTOFF.date()}; outcome is {CUTOFF.date()} to {END.date()}")

# Features from transactions before the cutoff only
past = df[df["InvoiceDate"] <= CUTOFF]
orders = past.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()
feat = orders.groupby("Customer ID").agg(
    last=("InvoiceDate", "max"),
    first=("InvoiceDate", "min"),
    frequency=("Invoice", "count"),
    monetary=("Revenue", "sum"),
)
feat["recency"] = (CUTOFF - feat["last"]).dt.days
feat["tenure"] = (CUTOFF - feat["first"]).dt.days
feat["avg_order"] = feat["monetary"] / feat["frequency"]

# Outcome: spend in the 90 days after the cutoff
future = df[df["InvoiceDate"] > CUTOFF].groupby("Customer ID")["Revenue"].sum()
feat["future_spend"] = future.reindex(feat.index).fillna(0)
feat["churned"] = (feat["future_spend"] == 0).astype(int)
print(f"\nCustomers at cutoff: {len(feat):,}")
print(f"Churn rate: {feat['churned'].mean():.1%}")

# Segments rebuilt from pre-cutoff features
X_rfm = StandardScaler().fit_transform(np.log1p(feat[["recency", "frequency", "monetary"]]))
feat["segment"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_rfm)

# Name segments by median recency: most recent = Champions, least recent = Lost
seg_order = feat.groupby("segment")["recency"].median().sort_values().index
names = ["Champions", "Recent", "At Risk", "Lost"]
feat["segment_name"] = feat["segment"].map({seg: names[i] for i, seg in enumerate(seg_order)})

profile = feat.groupby("segment_name").agg(
    customers=("recency", "size"),
    median_recency=("recency", "median"),
    median_frequency=("frequency", "median"),
    median_monetary=("monetary", "median"),
    churn_rate=("churned", "mean"),
    mean_future_spend=("future_spend", "mean"),
).round(2)
print("\nSegment outcomes:")
print(profile.sort_values("median_recency"))

# Churn model: tuned with CV, threshold chosen on training data only
features = ["recency", "frequency", "monetary", "tenure", "avg_order"]
X = feat[features]
y = feat["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

candidates = {
    "Logistic regression": (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10]},
    ),
    "Gradient boosting": (
        GradientBoostingClassifier(random_state=42),
        {"n_estimators": [100, 300], "max_depth": [2, 3], "learning_rate": [0.05, 0.1]},
    ),
}

results = {}
for name, (est, grid) in candidates.items():
    gs = GridSearchCV(est, grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    best = gs.best_estimator_
    rep = cross_val_score(best, X_tr, y_tr, scoring="roc_auc", cv=rcv, n_jobs=-1)

    oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 81)
    t_best = thresholds[int(np.argmax([f1_score(y_tr, oof >= t) for t in thresholds]))]

    proba = best.predict_proba(X_te)[:, 1]
    pred = (proba >= t_best).astype(int)
    results[name] = (best, t_best, rep.mean())

    print(f"\n=== {name} ===")
    print(f"Best params: {gs.best_params_}")
    print(f"Repeated CV ROC-AUC: {rep.mean():.3f} ± {rep.std():.3f}")
    print(f"Threshold (from training CV): {t_best:.2f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_te, proba):.3f}")
    print(f"Test recall: {recall_score(y_te, pred):.2f}, precision: {precision_score(y_te, pred):.2f}")

# Recency-only baseline: higher recency = higher churn, so score is +recency
print(f"\nRecency-only test ROC-AUC: {roc_auc_score(y_te, X_te['recency']):.3f}")

# Score every customer with out-of-fold predictions
winner = max(results, key=lambda n: results[n][2])
best, t, _ = results[winner]
print(f"\nBest model by repeated CV: {winner}")
feat["churn_probability"] = cross_val_predict(best, X, y, cv=cv, method="predict_proba")[:, 1]
feat["at_risk_flag"] = (feat["churn_probability"] >= t).astype(int)

feat.reset_index().to_csv(SCORES, index=False)
print(f"Saved customer scores to {SCORES}")

Features use data up to 2011-09-10; outcome is 2011-09-10 to 2011-12-09

Customers at cutoff: 5,248
Churn rate: 56.5%

Segment outcomes:
              customers  median_recency  median_frequency  median_monetary  \
segment_name                                                                 
Champions           867            19.0              14.0          5186.58   
Recent              658            25.5               2.0           587.30   
At Risk            1610           163.0               5.0          1450.72   
Lost               2113           326.0               1.0           294.88   

              churn_rate  mean_future_spend  
segment_name                                 
Champions           0.16            2082.35  
Recent              0.44             394.57  
At Risk             0.53             329.19  
Lost                0.80              82.22  

=== Logistic regression ===
Best params: {'logisticregression__C': 10}
Repeated CV ROC-AUC: 0.788 ± 0.017
Threshold (

In [6]:
# %% 4. Telco churn benchmark (separate customers)
tel = pd.read_csv(RAW_TELCO)
tel["TotalCharges"] = pd.to_numeric(tel["TotalCharges"].str.strip(), errors="coerce")
tel = tel.dropna(subset=["TotalCharges"])

y_t = (tel["Churn"] == "Yes").astype(int)
X_t = tel.drop(columns=["customerID", "Churn"])

num_cols = X_t.select_dtypes(include="number").columns.tolist()
cat_cols = X_t.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_t, y_t, test_size=0.2, stratify=y_t, random_state=42
)
cv_t = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rcv_t = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

candidates_t = {
    "Logistic regression": (
        Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "Gradient boosting": (
        Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(random_state=42))]),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [2, 3], "clf__learning_rate": [0.05, 0.1]},
    ),
}

for name, (pipe, grid) in candidates_t.items():
    gs = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv_t, n_jobs=-1).fit(X_tr_t, y_tr_t)
    best_t = gs.best_estimator_
    rep = cross_val_score(best_t, X_tr_t, y_tr_t, scoring="roc_auc", cv=rcv_t, n_jobs=-1)

    oof = cross_val_predict(best_t, X_tr_t, y_tr_t, cv=cv_t, method="predict_proba", n_jobs=-1)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 81)
    t_best = thresholds[int(np.argmax([f1_score(y_tr_t, oof >= t) for t in thresholds]))]

    proba = best_t.predict_proba(X_te_t)[:, 1]
    pred = (proba >= t_best).astype(int)

    print(f"\n=== {name} (Telco) ===")
    print(f"Best params: {gs.best_params_}")
    print(f"Repeated CV ROC-AUC: {rep.mean():.3f} ± {rep.std():.3f}")
    print(f"Threshold (from training CV): {t_best:.2f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_te_t, proba):.3f}")
    print(f"Test recall: {recall_score(y_te_t, pred):.2f}, precision: {precision_score(y_te_t, pred):.2f}")

print(f"\nTenure-only test ROC-AUC: {roc_auc_score(y_te_t, -X_te_t['tenure']):.3f}")



=== Logistic regression (Telco) ===
Best params: {'clf__C': 10}
Repeated CV ROC-AUC: 0.847 ± 0.008
Threshold (from training CV): 0.57
Test ROC-AUC: 0.834
Test recall: 0.74, precision: 0.52

=== Gradient boosting (Telco) ===
Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 2, 'clf__n_estimators': 100}
Repeated CV ROC-AUC: 0.850 ± 0.007
Threshold (from training CV): 0.30
Test ROC-AUC: 0.839
Test recall: 0.76, precision: 0.52

Tenure-only test ROC-AUC: 0.747


In [7]:
# %% 5. CLV as of the cutoff, expected revenue at risk, and holdout check
orders_cap = orders.copy()
orders_cap["Revenue_capped"] = orders_cap["Revenue"].clip(upper=ORDER_CAP)
capped_ids = orders.loc[orders["Revenue"] > ORDER_CAP, "Customer ID"].unique()
print(f"Invoices capped at £{ORDER_CAP:,}: {(orders['Revenue'] > ORDER_CAP).sum()}")

summary = summary_data_from_transaction_data(
    orders_cap, "Customer ID", "InvoiceDate",
    monetary_value_col="Revenue_capped",
    observation_period_end=CUTOFF, freq="W",
)
repeat = summary[summary["frequency"] > 0]
bgf = BetaGeoFitter(penalizer_coef=0.0).fit(summary["frequency"], summary["recency"], summary["T"])
ggf = GammaGammaFitter(penalizer_coef=0.0).fit(repeat["frequency"], repeat["monetary_value"])
summary.loc[summary["frequency"] == 0, "monetary_value"] = repeat["monetary_value"].mean()

summary["clv_orders_12m"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    52, summary["frequency"], summary["recency"], summary["T"]
)
summary["clv_order_value"] = ggf.conditional_expected_average_profit(
    summary["frequency"], summary["monetary_value"]
)
summary["CLV_12m_cutoff"] = summary["clv_orders_12m"] * summary["clv_order_value"]

summary.reset_index().to_csv(CLV_OUT, index=False)
print(f"Saved CLV table ({len(summary):,} customers) to {CLV_OUT}")

clv = summary[["CLV_12m_cutoff"]].reset_index()
clv["large_order_capped"] = clv["Customer ID"].isin(capped_ids).astype(int)

# Merge with churn scores; expected revenue at risk = churn probability x CLV
scores = pd.read_csv(SCORES).merge(clv, on="Customer ID", how="left")
scores["value_at_risk"] = scores["churn_probability"] * scores["CLV_12m_cutoff"]
scores.to_csv(FINAL, index=False)

print(f"\nCustomers scored: {len(scores):,}")
print(f"Total CLV as of cutoff: £{scores['CLV_12m_cutoff'].sum():,.0f}")
print(f"Total expected revenue at risk: £{scores['value_at_risk'].sum():,.0f}")
print(scores.groupby("segment_name").agg(
    Customers=("Customer ID", "size"),
    Avg_churn=("churn_probability", "mean"),
    Avg_CLV=("CLV_12m_cutoff", "mean"),
    Total_CLV=("CLV_12m_cutoff", "sum"),
    Revenue_at_risk=("value_at_risk", "sum"),
).round(2))

# Holdout check: fit on data before the last 90 days, test on those 90 days
inv_all = df.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()
ch = calibration_and_holdout_data(
    inv_all, "Customer ID", "InvoiceDate",
    calibration_period_end=CUTOFF, observation_period_end=END,
    freq="W", monetary_value_col="Revenue",
)
bgf_cal = BetaGeoFitter(penalizer_coef=0.0).fit(ch["frequency_cal"], ch["recency_cal"], ch["T_cal"])
ch["predicted_holdout"] = bgf_cal.predict(
    ch["duration_holdout"], ch["frequency_cal"], ch["recency_cal"], ch["T_cal"]
)
print("\nHoldout check (last 90 days):")
print(f"  Actual repeat orders:    {ch['frequency_holdout'].sum():,.0f}")
print(f"  Predicted repeat orders: {ch['predicted_holdout'].sum():,.0f}")
print(f"  Correlation: {ch[['frequency_holdout', 'predicted_holdout']].corr().iloc[0, 1]:.3f}")

Invoices capped at £50,000: 0
Saved CLV table (5,248 customers) to C:\Users\reinh\Downloads\Projects\Output CSVs\clv_table_cutoff.csv

Customers scored: 5,248
Total CLV as of cutoff: £7,148,023
Total expected revenue at risk: £1,805,196
              Customers  Avg_churn  Avg_CLV   Total_CLV  Revenue_at_risk
segment_name                                                            
At Risk            1610       0.53   836.57  1346871.01        597781.58
Champions           867       0.17  5285.73  4582725.90        502648.84
Lost               2113       0.79   277.62   586615.68        448303.46
Recent              658       0.44   960.20   631810.36        256461.68

Holdout check (last 90 days):
  Actual repeat orders:    4,616
  Predicted repeat orders: 3,632
  Correlation: 0.734


In [8]:
# %% 6. Data checks for the outlier (optional)
print(scores[scores["Customer ID"] == 16446].T)
print(df[df["Invoice"].astype(str) == "581483"][
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue", "InvoiceDate"]
])

                                   3608
Customer ID                       16446
last                2011-05-18 09:52:00
first               2011-05-18 09:52:00
frequency                             1
monetary                            2.9
recency                             115
tenure                              115
avg_order                           2.9
future_spend                        0.0
churned                               1
segment                               1
segment_name                       Lost
churn_probability              0.753394
at_risk_flag                          1
CLV_12m_cutoff               517.929285
large_order_capped                    0
value_at_risk                390.204866
Empty DataFrame
Columns: [Invoice, StockCode, Description, Quantity, Price, Revenue, InvoiceDate]
Index: []
